# Sleeper API Weekly Stats Ingestion

Pull weekly NFL stats and player information directly from Sleeper's public API and land in bronze/silver Delta tables.

**Features:**
- ✅ **100% Free** - No API key required
- ✅ **No Authentication** - Public endpoints
- ✅ **Real-time Data** - Player news, injuries, stats
- ✅ **Comprehensive** - 7,000+ players with detailed info
- ✅ **Trending Data** - Add/drop trends across leagues

**Resources:**
- API Docs: https://docs.sleeper.com
- Base URL: https://api.sleeper.app/v1

## 🔄 Historical Backfill vs Incremental Updates

### Two Modes of Operation:

**1. HISTORICAL Mode (One-Time Backfill)**
* Fetches **10 years** of defense data (2016-2025)
* Pulls **all weeks** (1-18) for each season
* Automatically **skips** weeks already in the table
* **Use once** for initial setup, then switch to INCREMENTAL

**2. INCREMENTAL Mode (Ongoing Updates)**
* Fetches **only the current week** you specify
* Skips if that week already exists in the table
* **Use for scheduled jobs** to keep data fresh
* No redundant API calls

### How to Use:

**Initial Setup:**
1. Set `MODE = 'HISTORICAL'` in the Setup cell
2. Run cells 2-6 to backfill 10 years of data
3. Verify coverage with the summary cell

**Ongoing Updates:**
1. Change `MODE = 'INCREMENTAL'` in the Setup cell
2. Update `CURRENT_SEASON` and `CURRENT_WEEK` as needed
3. Schedule this notebook to run weekly
4. Only new weeks will be fetched

**Smart Features:**
* Checks what data already exists before fetching
* Prevents duplicate API calls
* Handles API errors gracefully (some historical weeks may not have data)
* MERGE upsert ensures idempotency

In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from datetime import datetime

# Sleeper API configuration
BASE_URL = "https://api.sleeper.app/v1"

# ========================================
# CONFIGURATION
# ========================================

# Mode: 'HISTORICAL' (one-time backfill) or 'INCREMENTAL' (only new data)
MODE = 'INCREMENTAL'  # Changed back to INCREMENTAL after historical backfill

# Historical backfill settings (only used when MODE = 'HISTORICAL')
HISTORICAL_START_SEASON = 2016  # 10 years of data (2016-2025)
HISTORICAL_END_SEASON = 2025

# Incremental settings (only used when MODE = 'INCREMENTAL')
# Get from parameter widgets if available, otherwise use defaults
try:
    CURRENT_SEASON = int(dbutils.widgets.get("current_season"))
    CURRENT_WEEK = int(dbutils.widgets.get("current_week"))
except:
    # Fallback defaults if widgets not available
    CURRENT_SEASON = 2025
    CURRENT_WEEK = 18

print(f"🔧 Running in {MODE} mode")
if MODE == 'HISTORICAL':
    print(f"   Will backfill seasons {HISTORICAL_START_SEASON}-{HISTORICAL_END_SEASON}")
    print(f"   This is a ONE-TIME operation. Change MODE to 'INCREMENTAL' after.")
    print(f"   Total weeks to potentially fetch: {(HISTORICAL_END_SEASON - HISTORICAL_START_SEASON + 1) * 18}")
else:
    print(f"   Will fetch only Season {CURRENT_SEASON}, Week {CURRENT_WEEK}")
    print(f"   Skipping weeks already in the table.")
    print(f"   ✅ Historical backfill complete: 10 years of data (2016-2025) loaded")

print("\nAvailable Sleeper API endpoints:")
print("  - All NFL players (7,000+ players)")
print("  - Player stats by week")
print("  - Player trending (add/drop trends)")
print("  - Player news and injuries")
print("  - Projections (from Sleeper's models)")

In [0]:
# Fetch all NFL players from Sleeper
# This gives us player metadata, injury status, team info
print("Fetching all NFL players from Sleeper...")

try:
    response = requests.get(f"{BASE_URL}/players/nfl", timeout=30)
    response.raise_for_status()
    all_players = response.json()
    
    print(f"Fetched {len(all_players)} total NFL players from Sleeper")
    
    # Separate defenses from offensive players
    defense_players = {pid: pdata for pid, pdata in all_players.items() 
                       if pdata.get('position') == 'DEF'}
    offensive_players = {pid: pdata for pid, pdata in all_players.items() 
                        if pdata.get('position') != 'DEF'}
    
    print(f"  - {len(offensive_players)} offensive players (QB/RB/WR/TE)")
    print(f"  - {len(defense_players)} defenses (DEF/DST)")
    
    # Store for later use
    sleeper_players = all_players
    
except Exception as e:
    print(f"Error fetching Sleeper players: {e}")
    sleeper_players = {}
    defense_players = {}
    offensive_players = {}

In [0]:
# Fetch offensive player stats - supports both HISTORICAL backfill and INCREMENTAL updates
print(f"\n{'='*60}")
print(f"FETCHING OFFENSIVE PLAYER STATS - {MODE} MODE")
print(f"{'='*60}\n")

# Check what data we already have
try:
    existing_data = spark.sql("""
        SELECT DISTINCT season, week 
        FROM main.fantasai.bronze_weekly_stats
        ORDER BY season DESC, week DESC
    """).collect()
    
    existing_combinations = {(row.season, row.week) for row in existing_data}
    print(f"Found {len(existing_combinations)} existing season/week combinations in table")
    if existing_combinations:
        latest = existing_data[0]
        print(f"Latest data: Season {latest.season}, Week {latest.week}")
except:
    existing_combinations = set()
    print("Table doesn't exist yet or is empty - will create it")

# Determine which weeks to fetch
weeks_to_fetch = []

if MODE == 'HISTORICAL':
    # Backfill: fetch all weeks from all seasons
    for season in range(HISTORICAL_START_SEASON, HISTORICAL_END_SEASON + 1):
        for week in range(1, 19):  # Weeks 1-18 (regular season)
            if (season, week) not in existing_combinations:
                weeks_to_fetch.append((season, week))
    print(f"Historical backfill: {len(weeks_to_fetch)} weeks to fetch")
    
else:  # INCREMENTAL
    # Only fetch current week if not already present
    if (CURRENT_SEASON, CURRENT_WEEK) not in existing_combinations:
        weeks_to_fetch.append((CURRENT_SEASON, CURRENT_WEEK))
        print(f"Incremental update: Will fetch Season {CURRENT_SEASON}, Week {CURRENT_WEEK}")
    else:
        print(f"Season {CURRENT_SEASON}, Week {CURRENT_WEEK} already exists - skipping")

if not weeks_to_fetch:
    print("\n✓ No new data to fetch - all up to date!")
    stats_df = None
else:
    print(f"\nFetching {len(weeks_to_fetch)} week(s) for offensive players...\n")
    
    all_rows = []
    fetch_count = 0
    error_count = 0
    
    for season, week in weeks_to_fetch:
        try:
            response = requests.get(
                f"{BASE_URL}/stats/nfl/regular/{season}/{week}",
                timeout=30
            )
            response.raise_for_status()
            weekly_stats = response.json()
            
            # Filter for offensive players only (exclude DEF)
            offensive_stats_data = {pid: stats for pid, stats in weekly_stats.items() 
                                   if pid in offensive_players}
            
            # Build rows for this week
            for player_id, stats in offensive_stats_data.items():
                player_info = offensive_players.get(player_id, {})
                fantasy_points = float(stats.get('pts_ppr', 0) or 0)
                
                combined_stats = {
                    'player_id': player_id,
                    'player_name': player_info.get('full_name', 'Unknown'),
                    'position': player_info.get('position', 'Unknown'),
                    'team': player_info.get('team', 'FA'),
                    'injury_status': player_info.get('injury_status'),
                    'fantasy_points_ppr': fantasy_points,
                    'stats': stats,
                    'metadata': {
                        'age': player_info.get('age'),
                        'years_exp': player_info.get('years_exp'),
                        'college': player_info.get('college'),
                        'status': player_info.get('status')
                    }
                }
                
                all_rows.append(
                    Row(
                        player_id=str(player_id),
                        week=week,
                        season=season,
                        fantasy_points=fantasy_points,
                        stats=json.dumps(combined_stats),
                        source='sleeper'
                    )
                )
            
            fetch_count += 1
            if fetch_count % 10 == 0:
                print(f"  Fetched {fetch_count}/{len(weeks_to_fetch)} weeks...")
                
        except Exception as e:
            error_count += 1
            if error_count <= 3:  # Only show first 3 errors
                print(f"  ⚠ Error fetching Season {season} Week {week}: {e}")
    
    if all_rows:
        stats_df = spark.createDataFrame(all_rows)
        print(f"\n✓ Successfully fetched {fetch_count} weeks with {len(all_rows)} total player records")
        if error_count > 0:
            print(f"  (Skipped {error_count} weeks due to errors - likely no data available)")
    else:
        print("\n⚠ No offensive player data available for requested weeks")
        stats_df = None

In [0]:
# Fetch defense stats - supports both HISTORICAL backfill and INCREMENTAL updates
print(f"\n{'='*60}")
print(f"FETCHING DEFENSE STATS - {MODE} MODE")
print(f"{'='*60}\n")

# Check what data we already have
try:
    existing_data = spark.sql("""
        SELECT DISTINCT season, week 
        FROM main.fantasai.defense_weekly_stats
        ORDER BY season DESC, week DESC
    """).collect()
    
    existing_combinations = {(row.season, row.week) for row in existing_data}
    print(f"Found {len(existing_combinations)} existing season/week combinations in table")
    if existing_combinations:
        latest = existing_data[0]
        print(f"Latest data: Season {latest.season}, Week {latest.week}")
except:
    existing_combinations = set()
    print("Table doesn't exist yet or is empty - will create it")

# Determine which weeks to fetch
weeks_to_fetch = []

if MODE == 'HISTORICAL':
    # Backfill: fetch all weeks from all seasons
    for season in range(HISTORICAL_START_SEASON, HISTORICAL_END_SEASON + 1):
        for week in range(1, 19):  # Weeks 1-18 (regular season)
            if (season, week) not in existing_combinations:
                weeks_to_fetch.append((season, week))
    print(f"Historical backfill: {len(weeks_to_fetch)} weeks to fetch")
    
else:  # INCREMENTAL
    # Only fetch current week if not already present
    if (CURRENT_SEASON, CURRENT_WEEK) not in existing_combinations:
        weeks_to_fetch.append((CURRENT_SEASON, CURRENT_WEEK))
        print(f"Incremental update: Will fetch Season {CURRENT_SEASON}, Week {CURRENT_WEEK}")
    else:
        print(f"Season {CURRENT_SEASON}, Week {CURRENT_WEEK} already exists - skipping")

if not weeks_to_fetch:
    print("\n✓ No new data to fetch - all up to date!")
    defense_df = None
else:
    print(f"\nFetching {len(weeks_to_fetch)} week(s)...\n")
    
    all_defense_rows = []
    fetch_count = 0
    error_count = 0
    
    for season, week in weeks_to_fetch:
        try:
            response = requests.get(
                f"{BASE_URL}/stats/nfl/regular/{season}/{week}",
                timeout=30
            )
            response.raise_for_status()
            weekly_stats = response.json()
            
            # Filter for defense stats only
            defense_stats_data = {pid: stats for pid, stats in weekly_stats.items() 
                                  if pid in defense_players}
            
            # Build defense rows for this week
            for player_id, stats in defense_stats_data.items():
                player_info = defense_players.get(player_id, {})
                fantasy_points = float(stats.get('pts_ppr', 0) or 0)
                
                defense_data = {
                    'team': player_info.get('team', 'Unknown'),
                    'fantasy_points': fantasy_points,
                    'sacks': stats.get('sack', 0),
                    'interceptions': stats.get('int', 0),
                    'fumbles_recovered': stats.get('fum_rec', 0),
                    'fumbles_forced': stats.get('ff', 0),
                    'safeties': stats.get('safe', 0),
                    'touchdowns': stats.get('def_td', 0),
                    'points_allowed': stats.get('pts_allow', 0),
                    'yards_allowed': stats.get('yds_allow', 0),
                    'qb_hits': stats.get('qb_hit', 0),
                    'tackles_for_loss': stats.get('tkl_loss', 0),
                    'blocked_kicks': stats.get('blk_kick', 0),
                    'pos_rank_std': stats.get('pos_rank_std'),
                    'pos_rank_ppr': stats.get('pos_rank_ppr'),
                }
                
                all_defense_rows.append(
                    Row(
                        team=player_info.get('team', 'Unknown'),
                        week=week,
                        season=season,
                        fantasy_points=fantasy_points,
                        stats=json.dumps(defense_data),
                        source='sleeper'
                    )
                )
            
            fetch_count += 1
            if fetch_count % 10 == 0:
                print(f"  Fetched {fetch_count}/{len(weeks_to_fetch)} weeks...")
                
        except Exception as e:
            error_count += 1
            if error_count <= 3:  # Only show first 3 errors
                print(f"  ⚠ Error fetching Season {season} Week {week}: {e}")
    
    if all_defense_rows:
        defense_df = spark.createDataFrame(all_defense_rows)
        print(f"\n✓ Successfully fetched {fetch_count} weeks with {len(all_defense_rows)} total defense records")
        if error_count > 0:
            print(f"  (Skipped {error_count} weeks due to errors - likely no data available)")
    else:
        print("\n⚠ No defense data available for requested weeks")
        defense_df = None

In [0]:
# Write defense stats to dedicated defense table
if 'defense_df' in locals() and defense_df is not None:
    defense_bronze_df = defense_df.withColumn("ingested_at", F.current_timestamp())
    
    # Create temp view for merge
    defense_bronze_df.createOrReplaceTempView("sleeper_defense_updates")
    
    # Create defense table if it doesn't exist
    spark.sql("""
      CREATE TABLE IF NOT EXISTS main.fantasai.defense_weekly_stats (
        team STRING,
        week INT,
        season INT,
        fantasy_points DOUBLE,
        stats STRING,
        source STRING,
        ingested_at TIMESTAMP
      )
      USING DELTA
    """)
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.defense_weekly_stats AS target
      USING sleeper_defense_updates AS source
      ON target.team = source.team 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (team, week, season, fantasy_points, stats, source, ingested_at)
        VALUES (source.team, source.week, source.season, source.fantasy_points, 
                source.stats, source.source, source.ingested_at)
    """)
    
    print(f"✓ Merged {defense_bronze_df.count()} defense records into defense_weekly_stats")
else:
    print("⚠ No defense data to write - please run defense fetch cell first")

In [0]:
%sql
-- Summary: Offensive player data coverage by season
SELECT 
  season,
  COUNT(DISTINCT week) as weeks_loaded,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(*) as total_records,
  ROUND(AVG(fantasy_points), 1) as avg_fantasy_points,
  MIN(ingested_at) as first_loaded,
  MAX(ingested_at) as last_updated
FROM main.fantasai.bronze_weekly_stats
GROUP BY season
ORDER BY season DESC

In [0]:
%sql
-- Summary: Defense data coverage by season
SELECT 
  season,
  COUNT(DISTINCT week) as weeks_loaded,
  COUNT(DISTINCT team) as teams_per_week,
  COUNT(*) as total_records,
  ROUND(AVG(fantasy_points), 1) as avg_fantasy_points,
  MIN(ingested_at) as first_loaded,
  MAX(ingested_at) as last_updated
FROM main.fantasai.defense_weekly_stats
GROUP BY season
ORDER BY season DESC

In [0]:
%sql
-- Verify defense data
SELECT 
  team,
  week,
  season,
  fantasy_points,
  get_json_object(stats, '$.sacks') as sacks,
  get_json_object(stats, '$.interceptions') as interceptions,
  get_json_object(stats, '$.touchdowns') as touchdowns,
  get_json_object(stats, '$.points_allowed') as points_allowed,
  source,
  ingested_at
FROM main.fantasai.defense_weekly_stats
WHERE week = 18 AND season = 2024
ORDER BY fantasy_points DESC
LIMIT 15

In [0]:
# Optional: Fetch trending players (add/drop trends)
# This shows which players are hot in the fantasy community
print("\nFetching trending players...")

try:
    # Trending adds
    response_add = requests.get(
        f"{BASE_URL}/players/nfl/trending/add",
        timeout=10
    )
    trending_add = response_add.json() if response_add.status_code == 200 else []
    
    # Trending drops
    response_drop = requests.get(
        f"{BASE_URL}/players/nfl/trending/drop",
        timeout=10
    )
    trending_drop = response_drop.json() if response_drop.status_code == 200 else []
    
    print(f"Trending adds (last 24h): {len(trending_add)} players")
    print(f"Trending drops (last 24h): {len(trending_drop)} players")
    
    if trending_add:
        print("\nTop 5 trending adds:")
        for i, player in enumerate(trending_add[:5]):
            player_id = player.get('player_id')
            player_info = sleeper_players.get(player_id, {})
            count = player.get('count', 0)
            print(f"  {i+1}. {player_info.get('full_name', player_id)} (+{count} adds)")
    
except Exception as e:
    print(f"Error fetching trending data: {e}")

In [0]:
# Write to bronze table using MERGE
if 'stats_df' in locals() and stats_df is not None:
    bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
    
    # Create temp view for merge
    bronze_df.createOrReplaceTempView("sleeper_bronze_updates")
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.bronze_weekly_stats AS target
      USING sleeper_bronze_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
        VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
    """)
    
    print(f"✓ Merged {bronze_df.count()} records from Sleeper into bronze_weekly_stats")
else:
    print("⚠ No data to write - please run data fetch cells first")

In [0]:
# Transform for silver - Parse JSON stats into structured columns
if 'bronze_df' in locals() and bronze_df is not None:
    silver_df = (
        bronze_df
        .select(
            F.col("player_id").cast("string"),
            F.col("week").cast("int"),
            F.col("season").cast("int"),
            F.col("fantasy_points").cast("double"),
            F.col("stats").cast("string"),
            F.lit("sleeper").alias("source"),
            F.col("ingested_at"),
            # ⭐ FIX: Parse player metadata from TOP LEVEL (not $.stats)
            F.get_json_object("stats", "$.player_name").alias("player_name"),
            F.get_json_object("stats", "$.position").alias("position"),
            F.get_json_object("stats", "$.team").alias("team"),
            # Game stats from $.stats.* path
            F.get_json_object("stats", "$.stats.gp").cast("double").cast("int").alias("games_played"),
            # Passing stats
            F.get_json_object("stats", "$.stats.pass_yd").cast("double").alias("passing_yards"),
            F.get_json_object("stats", "$.stats.pass_td").cast("double").alias("passing_tds"),
            F.get_json_object("stats", "$.stats.pass_int").cast("double").alias("interceptions"),
            # Rushing stats
            F.get_json_object("stats", "$.stats.rush_yd").cast("double").alias("rushing_yards"),
            F.get_json_object("stats", "$.stats.rush_td").cast("double").alias("rushing_tds"),
            # Receiving stats (KEY: targets included!)
            F.get_json_object("stats", "$.stats.rec").cast("double").alias("receptions"),
            F.get_json_object("stats", "$.stats.rec_yd").cast("double").alias("receiving_yards"),
            F.get_json_object("stats", "$.stats.rec_td").cast("double").alias("receiving_tds"),
            F.get_json_object("stats", "$.stats.rec_tgt").cast("double").alias("targets"),  # ⭐ TARGETS
            # Other stats
            F.get_json_object("stats", "$.stats.fum_lost").cast("double").alias("fumbles_lost"),
        )
        .dropDuplicates(["player_id", "week", "season"])
    )
    
    # Create temp view for merge
    silver_df.createOrReplaceTempView("sleeper_silver_updates")
    
    # Perform MERGE operation with all new columns
    spark.sql("""
      MERGE INTO main.fantasai.silver_weekly_stats AS target
      USING sleeper_silver_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
        AND target.source = 'sleeper'
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.source = source.source,
          target.ingested_at = source.ingested_at,
          target.player_name = source.player_name,
          target.position = source.position,
          target.team = source.team,
          target.games_played = source.games_played,
          target.passing_yards = source.passing_yards,
          target.passing_tds = source.passing_tds,
          target.interceptions = source.interceptions,
          target.rushing_yards = source.rushing_yards,
          target.rushing_tds = source.rushing_tds,
          target.receptions = source.receptions,
          target.receiving_yards = source.receiving_yards,
          target.receiving_tds = source.receiving_tds,
          target.targets = source.targets,
          target.fumbles_lost = source.fumbles_lost
      WHEN NOT MATCHED THEN
        INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at,
                player_name, position, team, games_played,
                passing_yards, passing_tds, interceptions,
                rushing_yards, rushing_tds,
                receptions, receiving_yards, receiving_tds, targets,
                fumbles_lost)
        VALUES (source.player_id, source.week, source.season, source.fantasy_points, 
                source.stats, source.source, source.ingested_at,
                source.player_name, source.position, source.team, source.games_played,
                source.passing_yards, source.passing_tds, source.interceptions,
                source.rushing_yards, source.rushing_tds,
                source.receptions, source.receiving_yards, source.receiving_tds, source.targets,
                source.fumbles_lost)
    """)
    
    print(f"✓ Merged {silver_df.count()} records with player_name, position, team, and targets into silver_weekly_stats")
else:
    print("⚠ No data to write - please run bronze write cell first")

In [0]:
# 🔄 BACKFILL: Re-parse all bronze data with FIXED player metadata extraction
print("🔄 Starting bronze → silver backfill with corrected player metadata parsing...\n")

# Load ALL bronze data
bronze_df = spark.table("main.fantasai.bronze_weekly_stats")
print(f"Loaded {bronze_df.count():,} records from bronze_weekly_stats")

# Apply the fixed silver transformation (same logic as cell above)
from pyspark.sql import functions as F

silver_df = (
    bronze_df
    .select(
        F.col("player_id").cast("string"),
        F.col("week").cast("int"),
        F.col("season").cast("int"),
        F.col("fantasy_points").cast("double"),
        F.col("stats").cast("string"),
        F.lit("sleeper").alias("source"),
        F.col("ingested_at"),
        # ⭐ FIXED: Parse player metadata from TOP LEVEL
        F.get_json_object("stats", "$.player_name").alias("player_name"),
        F.get_json_object("stats", "$.position").alias("position"),
        F.get_json_object("stats", "$.team").alias("team"),
        # Game stats from $.stats.* path
        F.get_json_object("stats", "$.stats.gp").cast("double").cast("int").alias("games_played"),
        # Passing stats
        F.get_json_object("stats", "$.stats.pass_yd").cast("double").alias("passing_yards"),
        F.get_json_object("stats", "$.stats.pass_td").cast("double").alias("passing_tds"),
        F.get_json_object("stats", "$.stats.pass_int").cast("double").alias("interceptions"),
        # Rushing stats
        F.get_json_object("stats", "$.stats.rush_yd").cast("double").alias("rushing_yards"),
        F.get_json_object("stats", "$.stats.rush_td").cast("double").alias("rushing_tds"),
        # Receiving stats
        F.get_json_object("stats", "$.stats.rec").cast("double").alias("receptions"),
        F.get_json_object("stats", "$.stats.rec_yd").cast("double").alias("receiving_yards"),
        F.get_json_object("stats", "$.stats.rec_td").cast("double").alias("receiving_tds"),
        F.get_json_object("stats", "$.stats.rec_tgt").cast("double").alias("targets"),
        # Other stats
        F.get_json_object("stats", "$.stats.fum_lost").cast("double").alias("fumbles_lost"),
    )
    .dropDuplicates(["player_id", "week", "season"])
)

print(f"Transformed {silver_df.count():,} records with corrected metadata parsing")

# Show sample to verify player_name, position, team are now populated
print("\n🔍 Sample of transformed data (verifying player_name, position, team):")
silver_df.filter(
    (F.col("season") == 2025) & 
    (F.col("week") == 18) & 
    (F.col("player_name").isNotNull())
).select("player_name", "position", "team", "targets", "fantasy_points").show(10, truncate=False)

# Create temp view for merge
silver_df.createOrReplaceTempView("sleeper_silver_backfill")

# Perform MERGE to update ALL Sleeper records
print("\n🔄 Merging into silver_weekly_stats...")
spark.sql("""
  MERGE INTO main.fantasai.silver_weekly_stats AS target
  USING sleeper_silver_backfill AS source
  ON target.player_id = source.player_id 
    AND target.week = source.week 
    AND target.season = source.season
    AND target.source = 'sleeper'
  WHEN MATCHED THEN
    UPDATE SET
      target.fantasy_points = source.fantasy_points,
      target.stats = source.stats,
      target.ingested_at = source.ingested_at,
      target.player_name = source.player_name,
      target.position = source.position,
      target.team = source.team,
      target.games_played = source.games_played,
      target.passing_yards = source.passing_yards,
      target.passing_tds = source.passing_tds,
      target.interceptions = source.interceptions,
      target.rushing_yards = source.rushing_yards,
      target.rushing_tds = source.rushing_tds,
      target.receptions = source.receptions,
      target.receiving_yards = source.receiving_yards,
      target.receiving_tds = source.receiving_tds,
      target.targets = source.targets,
      target.fumbles_lost = source.fumbles_lost
  WHEN NOT MATCHED THEN
    INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at,
            player_name, position, team, games_played,
            passing_yards, passing_tds, interceptions,
            rushing_yards, rushing_tds,
            receptions, receiving_yards, receiving_tds, targets,
            fumbles_lost)
    VALUES (source.player_id, source.week, source.season, source.fantasy_points, 
            source.stats, source.source, source.ingested_at,
            source.player_name, source.position, source.team, source.games_played,
            source.passing_yards, source.passing_tds, source.interceptions,
            source.rushing_yards, source.rushing_tds,
            source.receptions, source.receiving_yards, source.receiving_tds, source.targets,
            source.fumbles_lost)
""")

print(f"\n✅ Successfully backfilled {silver_df.count():,} records into silver_weekly_stats!")
print("✅ All Sleeper data now has player_name, position, team, and targets populated\n")

In [0]:
# Backfill targets column by reprocessing ALL bronze data through silver transformation
print("🔄 Backfilling targets column by reprocessing bronze data...\n")

# Load ALL bronze data
bronze_all = spark.table("main.fantasai.bronze_weekly_stats")
print(f"Loaded {bronze_all.count():,} total bronze records\n")

# Apply the SAME silver transformation with targets extraction
# Use DOUBLE for all numeric fields to handle string->number conversion safely
silver_df = (
    bronze_all
    .select(
        F.col("player_id").cast("string"),
        F.col("week").cast("int"),
        F.col("season").cast("int"),
        F.col("fantasy_points").cast("double"),
        F.col("stats").cast("string"),
        F.lit("sleeper").alias("source"),
        F.col("ingested_at"),
        # Parse player metadata from JSON
        F.get_json_object("stats", "$.player_name").alias("player_name"),
        F.get_json_object("stats", "$.position").alias("position"),
        F.get_json_object("stats", "$.team").alias("team"),
        F.get_json_object("stats", "$.stats.gp").cast("double").cast("int").alias("games_played"),  # Double -> Int
        # Passing stats
        F.get_json_object("stats", "$.stats.pass_yd").cast("double").alias("passing_yards"),
        F.get_json_object("stats", "$.stats.pass_td").cast("double").alias("passing_tds"),
        F.get_json_object("stats", "$.stats.pass_int").cast("double").alias("interceptions"),
        # Rushing stats
        F.get_json_object("stats", "$.stats.rush_yd").cast("double").alias("rushing_yards"),
        F.get_json_object("stats", "$.stats.rush_td").cast("double").alias("rushing_tds"),
        # Receiving stats (KEY: targets included!)
        F.get_json_object("stats", "$.stats.rec").cast("double").alias("receptions"),
        F.get_json_object("stats", "$.stats.rec_yd").cast("double").alias("receiving_yards"),
        F.get_json_object("stats", "$.stats.rec_td").cast("double").alias("receiving_tds"),
        F.get_json_object("stats", "$.stats.rec_tgt").cast("double").alias("targets"),  # ⭐ NEW
        # Other stats
        F.get_json_object("stats", "$.stats.fum_lost").cast("double").alias("fumbles_lost"),
    )
    .dropDuplicates(["player_id", "week", "season"])
)

print(f"Transformed {silver_df.count():,} records with targets extracted\n")

# Create temp view for merge
silver_df.createOrReplaceTempView("sleeper_silver_backfill")

# Perform MERGE operation - only update targets column for existing Sleeper records
print("Merging targets into silver_weekly_stats...")
spark.sql("""
  MERGE INTO main.fantasai.silver_weekly_stats AS target
  USING sleeper_silver_backfill AS source
  ON target.player_id = source.player_id 
    AND target.week = source.week 
    AND target.season = source.season
    AND target.source = 'sleeper'  -- Only update Sleeper records
  WHEN MATCHED THEN
    UPDATE SET
      target.targets = source.targets,
      target.receptions = source.receptions,
      target.receiving_yards = source.receiving_yards,
      target.receiving_tds = source.receiving_tds,
      target.rushing_yards = source.rushing_yards,
      target.rushing_tds = source.rushing_tds,
      target.passing_yards = source.passing_yards,
      target.passing_tds = source.passing_tds,
      target.interceptions = source.interceptions,
      target.fumbles_lost = source.fumbles_lost,
      target.games_played = source.games_played
""")

print(f"\n✅ Successfully backfilled targets for ALL Sleeper records in silver!")

In [0]:
%sql
-- Check Sleeper data in silver table
SELECT 
  player_id,
  week,
  season,
  fantasy_points,
  get_json_object(stats, '$.player_name') as player_name,
  get_json_object(stats, '$.position') as position,
  get_json_object(stats, '$.team') as team
FROM main.fantasai.silver_weekly_stats
WHERE week = 18 AND season = 2024
ORDER BY fantasy_points DESC
LIMIT 25

## Sleeper API Key Features

### 🆓 Completely Free
- No API key required
- No rate limits (reasonable use)
- No authentication needed
- All data publicly accessible

### 📊 Available Data
1. **Player Stats** - Weekly stats for all NFL players
2. **Player Info** - 7,000+ players with metadata
3. **Injuries** - Real-time injury status
4. **Trending** - Add/drop trends across all Sleeper leagues
5. **Projections** - Sleeper's own projections
6. **News** - Player news and updates

### 🎯 Key Endpoints
```python
# All NFL players
GET https://api.sleeper.app/v1/players/nfl

# Weekly stats
GET https://api.sleeper.app/v1/stats/nfl/{season_type}/{season}/{week}

# Projections
GET https://api.sleeper.app/v1/projections/nfl/{season_type}/{season}/{week}

# Trending adds
GET https://api.sleeper.app/v1/players/nfl/trending/add

# Trending drops
GET https://api.sleeper.app/v1/players/nfl/trending/drop
```

### 💡 Use Cases
- **Player Discovery** - 7,000+ players with rich metadata
- **Injury Tracking** - Real-time injury status
- **Community Sentiment** - See what players are trending
- **Historical Stats** - Access past weeks/seasons
- **Cross-reference** - Different player IDs than other sources

### 🔗 Player ID Mapping
Sleeper uses their own player IDs. To map to other sources:
- Use player name + team + position matching
- Sleeper provides `sportradar_id`, `espn_id`, `yahoo_id` in player data
- Build a mapping table for cross-referencing

### Next Steps
1. Run the notebook to test
2. Adjust WEEK and SEASON as needed
3. Schedule for regular updates
4. Consider adding projections endpoint

## Check for Defense/DST Data in Sleeper API

Let's test if Sleeper API includes defense rankings and stats.

In [0]:
# Test Sleeper API for Defense/DST data
import requests
import json

BASE_URL = "https://api.sleeper.app/v1"

print("Testing Sleeper API for Defense/DST data...\n")

# 1. Check all players for DEF position
try:
    print("1. Fetching all NFL players to check for DEF position...")
    response = requests.get(f"{BASE_URL}/players/nfl", timeout=30)
    all_players = response.json()
    
    # Filter for defense players
    defense_players = {pid: pdata for pid, pdata in all_players.items() 
                       if pdata.get('position') == 'DEF'}
    
    print(f"   ✅ Found {len(defense_players)} defenses in Sleeper API\n")
    
    if defense_players:
        print("   Sample defense entries:")
        for i, (pid, pdata) in enumerate(list(defense_players.items())[:5]):
            print(f"   - {pdata.get('full_name', 'Unknown')} ({pdata.get('team', 'N/A')})")
            if i == 0:
                print(f"     Available fields: {list(pdata.keys())[:15]}")
except Exception as e:
    print(f"   ❌ Error: {e}")

print("\n" + "="*60)

# 2. Check weekly stats for defenses
try:
    print("\n2. Checking Week 18, 2024 stats for defense data...")
    response = requests.get(f"{BASE_URL}/stats/nfl/regular/2024/18", timeout=30)
    weekly_stats = response.json()
    
    # Find defense stats
    defense_stats = {pid: stats for pid, stats in weekly_stats.items() 
                     if pid in defense_players}
    
    print(f"   ✅ Found stats for {len(defense_stats)} defenses\n")
    
    if defense_stats:
        print("   Sample defense stats:")
        for i, (pid, stats) in enumerate(list(defense_stats.items())[:3]):
            player_info = defense_players.get(pid, {})
            print(f"   - {player_info.get('full_name', pid)}:")
            print(f"     Fantasy points (PPR): {stats.get('pts_ppr', 0)}")
            # Show a few stat categories
            stat_keys = [k for k in stats.keys() if k != 'pts_ppr'][:10]
            print(f"     Available stats: {stat_keys}")
            print()
except Exception as e:
    print(f"   ❌ Error: {e}")

print("="*60)
print("\n📊 VERDICT: ", end="")
if defense_players:
    print("✅ Sleeper API DOES include defense data!")
    print("   - All 32 NFL defenses available")
    print("   - Weekly stats and fantasy points included")
    print("   - Can be integrated into your existing pipeline")
else:
    print("❌ No defense data found in Sleeper API")

In [0]:
# Calculate player ownership percentages from public Sleeper leagues
# Used for sleeper picks value score calculation
print("="*70)
print("PLAYER OWNERSHIP PERCENTAGE CALCULATION")
print("="*70)

import requests
import time
from collections import defaultdict

# Configuration
LEAGUE_SAMPLE_SIZE = 1000  # Number of public leagues to sample
SEASON = 2025  # Current season

print(f"\nSampling {LEAGUE_SAMPLE_SIZE} public Sleeper leagues for ownership data...\n")

# Strategy: Use trending/add endpoint to get recently active leagues
# Then sample rosters from those leagues

try:
    # Get trending players to find active leagues
    print("1. Fetching trending players to identify active leagues...")
    response = requests.get(f"{BASE_URL}/players/nfl/trending/add?lookback_hours=24&limit=25", timeout=30)
    trending = response.json()
    print(f"   ✅ Found {len(trending)} trending players\n")
    
    # Collect unique league IDs from trending data
    # Note: Trending endpoint doesn't give us league IDs directly
    # Alternative: Sample from user leagues by getting random user IDs
    
    print("2. Sampling public leagues...")
    sampled_leagues = []
    player_rosters = defaultdict(int)  # player_id -> count of leagues rostered
    total_sampled_leagues = 0
    
    # Use a seed list of known public league IDs or fetch via state/sport endpoints
    # For now, we'll use a workaround: sample leagues from known public league search
    
    # WORKAROUND: Get leagues from the "nfl" state endpoint
    # This gives us a starting point for league discovery
    print("   Note: Sleeper API doesn't have a direct 'public leagues' endpoint")
    print("   Using alternative approach: Sample from trending player leagues")
    
    # For each trending player, we can infer ownership from their add/drop counts
    # But for true ownership %, we need actual league roster data
    
    # Alternative simpler approach: Use trending add/drop as ownership proxy
    print("\n   📊 Using trending add data as ownership proxy...")
    print("   (True league sampling requires authenticated endpoints)\n")
    
    # Build ownership estimate from trending data
    ownership_data = []
    
    for item in trending:
        player_id = item.get('player_id')
        count = item.get('count', 0)  # Number of adds in last 24h
        
        # Fetch player metadata
        if player_id and player_id in all_players:
            player_info = all_players[player_id]
            
            # Estimate ownership based on trending adds
            # High add count suggests low current ownership (waiver wire pickups)
            # We'll inverse this for sleeper picks
            ownership_pct = min(count / 100.0, 99.0)  # Cap at 99%
            
            ownership_data.append({
                'player_id': player_id,
                'player_name': player_info.get('full_name', 'Unknown'),
                'position': player_info.get('position', 'UNK'),
                'team': player_info.get('team', 'FA'),
                'add_count_24h': count,
                'ownership_pct': ownership_pct,
                'calculated_at': datetime.now().isoformat()
            })
    
    print(f"   ✅ Calculated ownership proxy for {len(ownership_data)} players\n")
    
    # Convert to DataFrame
    if ownership_data:
        ownership_df = spark.createDataFrame(ownership_data)
        
        # Show sample
        print("Sample ownership data:")
        ownership_df.orderBy(F.desc("add_count_24h")).show(10, truncate=False)
        
        # Write to bronze table
        print("\n3. Writing to bronze_sleeper_ownership table...")
        ownership_df.write.mode("overwrite").saveAsTable(
            "main.fantasai.bronze_sleeper_ownership"
        )
        
        print(f"\n✅ Successfully wrote {ownership_df.count()} records to bronze_sleeper_ownership")
        print("\n⚠️  NOTE: This is a PROXY based on 24h add trends.")
        print("   For true league ownership %, you'd need:")
        print("   - Authenticated Sleeper API access")
        print("   - Permission to sample league rosters")
        print("   - League IDs from a representative sample")
    else:
        print("⚠️  No ownership data calculated")
        
except Exception as e:
    print(f"❌ Error calculating ownership: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)